
### identify the node (env/node/rssi)

two dataset strategy:

a. data from all env -> into a dataset and split to train 0.75/test 0.25

b. data from 3 env (1, 2, 3) -> for train / 1 env -> test (4)


500 seq length as frame

model: ResNet

overlapping: 40%

report:

model.summary() 

hyper parameter (overlapping / learning rate = 0.001)

epoch 20

In [15]:
# 📓 Jupyter Notebook: Preprocess e1-n0 dataset

import pandas as pd
import numpy as np

# === Step 1: Load CSV ===
df = pd.read_csv("./env/e4-garden.csv")  # your dataset path here

# === Step 2: time series sorting ===
df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values('ts')

# === Step 3: set environment ===
df['env'] = 1

# === Step 4: filter for e1-n0 ===
df_n0 = df[df['device'] == 'RIOT-BLE-0'].copy()

# === Step 5: diff ===
df_n0['rssi_diff'] = df_n0['rssi'].diff()

# === Step 6: normalization (min-max) ===
y_min = df_n0['rssi_diff'].min()
y_max = df_n0['rssi_diff'].max()
df_n0['rssi_norm'] = (df_n0['rssi_diff'] - y_min) / (y_max - y_min)

# === Step 7: create timeslot（100 packets）===
# time series sorting
df_n0 = df_n0.sort_values('ts').reset_index(drop=True)
# 100 packets per timeslot
WINDOW_SIZE = 100
df_n0['timeslot'] = df_n0.index // WINDOW_SIZE
# === Step 8: unify column names ===
df_n0['node'] = 'n0'
df_n0_final = df_n0[['timeslot', 'env', 'node', 'rssi_norm']].rename(columns={'rssi_norm': 'rssi'})

# === Step 9: display results ===
df_n0_final.head(101)

,timeslot,env,node,rssi
0,0,1,n0,NaN
1,0,1,n0,0.666667
2,0,1,n0,0.481481
3,0,1,n0,0.592593
4,0,1,n0,0.370370
...,...,...,...,...
96,0,1,n0,0.518519
97,0,1,n0,0.518519
98,0,1,n0,0.370370
99,0,1,n0,0.481481


In [16]:
import pandas as pd
import numpy as np
import glob

# parameters
WINDOW_SIZE = 500 # 100/500/1000
STRIDE = 40 # 40/50

# === file paths ===
env_files = [
    "env/e0-bridge.csv",
    "env/e1-lake.csv",
    "env/e2-forest.csv",
    "env/e3-river.csv",
    "env/e4-garden.csv"
]

# === device mapping ===
device_to_label = {
    "RIOT-BLE-0": 0,
    "RIOT-BLE-1": 1,
    "RIOT-BLE-2": 2,
    "RIOT-BLE-3": 3
}

# === store data ===
X = []
y = []
env_ids = []

# each environment
for env_id, file in enumerate(env_files):

    df = pd.read_csv(file)

    # time series sorting
    df['ts'] = pd.to_datetime(df['ts'])
    df = df.sort_values('ts')

    # process each device
    for device, label in device_to_label.items():

        df_node = df[df['device'] == device].copy()
        if len(df_node) < WINDOW_SIZE:
            continue

        # === diff ===
        df_node['rssi_diff'] = df_node['rssi'].diff()
        
        # === drop NaN ===
        df_node = df_node.dropna(subset=['rssi_diff'])
        # === normalization (based on each node) ===
        y_min = df_node['rssi_diff'].min()
        y_max = df_node['rssi_diff'].max()

        if y_max - y_min == 0:
            continue

        df_node['rssi_norm'] = (df_node['rssi_diff'] - y_min) / (y_max - y_min)

        # === turn to numpy ===
        data = df_node['rssi_norm'].values

        # === create sequence ===
        for i in range(0, len(data) - WINDOW_SIZE, STRIDE):
            seq = data[i:i+WINDOW_SIZE]

            X.append(seq)
            y.append(label)
            env_ids.append(env_id)

# === turn to numpy ===
X = np.array(X)
y = np.array(y)
env_ids = np.array(env_ids)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("env_ids shape:", env_ids.shape)

X = X.astype(np.float32)
y = y.astype(np.int64)
env_ids = env_ids.astype(np.int64)

# PyTorch Conv1d input: (batch, channels, length)
X_data = X[:, np.newaxis, :]  
print("X_data shape:", X_data.shape)


X shape: (8277, 500)
y shape: (8277,)
env_ids shape: (8277,)
X_data shape: (8277, 1, 500)


In [17]:
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

# =========================
# Version 1: random split
# =========================
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_data, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# turn into PyTorch tensors
X_train_r_tensor = torch.tensor(X_train_r, dtype=torch.float32)
X_test_r_tensor  = torch.tensor(X_test_r, dtype=torch.float32)
y_train_r_tensor = torch.tensor(y_train_r, dtype=torch.long)
y_test_r_tensor  = torch.tensor(y_test_r, dtype=torch.long)

# create Dataset and DataLoader
train_dataset_r = TensorDataset(X_train_r_tensor, y_train_r_tensor)
test_dataset_r  = TensorDataset(X_test_r_tensor, y_test_r_tensor)

train_loader_r = DataLoader(train_dataset_r, batch_size=64, shuffle=True)
test_loader_r  = DataLoader(test_dataset_r, batch_size=64, shuffle=False)

print("\n=== Version 1: Random Split ===")
print("X_train:", X_train_r_tensor.shape)
print("X_test :", X_test_r_tensor.shape)
print("y_train:", y_train_r_tensor.shape)
print("y_test :", y_test_r_tensor.shape)


=== Version 1: Random Split ===
X_train: torch.Size([6207, 1, 500])
X_test : torch.Size([2070, 1, 500])
y_train: torch.Size([6207])
y_test : torch.Size([2070])


In [18]:
# =========================
# Version 2: env-based split
# =========================
train_envs = [1, 2, 3]   
test_env = 4           

train_mask = np.isin(env_ids, train_envs)
test_mask = (env_ids == test_env)

X_train_e = X_data[train_mask]
y_train_e = y[train_mask]

X_test_e = X_data[test_mask]
y_test_e = y[test_mask]

# turn into PyTorch tensors
X_train_e_tensor = torch.tensor(X_train_e, dtype=torch.float32)
X_test_e_tensor  = torch.tensor(X_test_e, dtype=torch.float32)
y_train_e_tensor = torch.tensor(y_train_e, dtype=torch.long)
y_test_e_tensor  = torch.tensor(y_test_e, dtype=torch.long)

# create Dataset and DataLoader
train_dataset_e = TensorDataset(X_train_e_tensor, y_train_e_tensor)
test_dataset_e  = TensorDataset(X_test_e_tensor, y_test_e_tensor)

train_loader_e = DataLoader(train_dataset_e, batch_size=64, shuffle=True)
test_loader_e  = DataLoader(test_dataset_e, batch_size=64, shuffle=False)

print("\n=== Version 2: Env-based Split ===")
print("Train envs:", train_envs)
print("Test env :", test_env)
print("X_train:", X_train_e_tensor.shape)
print("X_test :", X_test_e_tensor.shape)
print("y_train:", y_train_e_tensor.shape)
print("y_test :", y_test_e_tensor.shape)


=== Version 2: Env-based Split ===
Train envs: [1, 2, 3]
Test env : 4
X_train: torch.Size([4793, 1, 500])
X_test : torch.Size([1538, 1, 500])
y_train: torch.Size([4793])
y_test : torch.Size([1538])


In [19]:
# === Residual Block ===
import torch.nn as nn

class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm1d(out_channels) # batch norm after conv1
        self.relu = nn.ReLU(inplace=True)

        # self.dropout = nn.Dropout(p=0.1) # dropout layer

        self.conv2 = nn.Conv1d(
            out_channels, out_channels,
            kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm1d(out_channels) # batch norm after conv2

        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels: # projection shortcut if dimensions differ
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
    
    # forward pass
    def forward(self, x):
        identity = self.shortcut(x)

        # conv1 -> bn -> relu
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        # out = self.dropout(out) # dropout 

        # conv2 -> bn
        out = self.conv2(out)
        out = self.bn2(out)

        # add shortcut
        out += identity
        out = self.relu(out)

        return out

In [20]:
# === ResNet1D Model ===

class ResNet1D(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # initial convolution and pooling
        self.stem = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )

        # layer1
        self.layer1 = nn.Sequential(
            ResidualBlock1D(16, 16, stride=1)
        )

        # layer2
        self.layer2 = nn.Sequential(
            ResidualBlock1D(16, 32, stride=2)
        )

        # layer3
        self.layer3 = nn.Sequential(
            ResidualBlock1D(32, 64, stride=2)
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        # self.dropout = nn.Dropout(p=0.3)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.stem(x)         # (B, 1, 100) -> (B, 16, 25)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.global_pool(x)  # (B, 64, 1)
        x = x.squeeze(-1)        # (B, 64, 1) -> (B, 64)
        # x = self.dropout(x)      # dropout
        x = self.fc(x)           # (B, num_classes)
        return x

num_classes = len(np.unique(y))
model = ResNet1D(num_classes=num_classes)
print(model)

ResNet1D(
  (stem): Sequential(
    (0): Conv1d(1, 16, kernel_size=(7,), stride=(2,), padding=(3,), bias=False)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool1d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): ResidualBlock1D(
      (conv1): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
  )
  (layer2): Sequential(
    (0): ResidualBlock1D(
      (conv1): Conv1d(16, 32, kernel_size=(3,), stride=(2,), padding=(1,), bias=False)
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=T

In [21]:
# =========================
# Step 3: training setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [22]:
# =========================
# Step 4: training loop
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)
            loss = criterion(outputs, yb)

            total_loss += loss.item() * xb.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return total_loss / total, correct / total

In [ ]:
# =========================
# Step 5: run training
# =========================
num_epochs = 20


def train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, device, num_epochs):
      best_test_loss = float("inf")
      best_state = None
      for epoch in range(num_epochs):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            test_loss, test_acc = evaluate(model, test_loader, criterion, device)

            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                  f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

      # save best model
      if test_loss < best_test_loss:
          best_test_loss = test_loss
          best_state = model.state_dict()
      model.load_state_dict(best_state)

# ===========================
# version 1: random split
# ===========================

print("\n=== Version 1: random split ===\n")
train_and_evaluate(model, train_loader_r, test_loader_r, criterion, optimizer, device, num_epochs)


=== Version 1: random split ===

Epoch [1/20] Train Loss: 1.0208, Train Acc: 0.5421 | Test Loss: 0.8456, Test Acc: 0.6014
Epoch [2/20] Train Loss: 0.7316, Train Acc: 0.7077 | Test Loss: 0.6851, Test Acc: 0.7169
Epoch [3/20] Train Loss: 0.5377, Train Acc: 0.8112 | Test Loss: 0.6123, Test Acc: 0.7256
Epoch [4/20] Train Loss: 0.3928, Train Acc: 0.8652 | Test Loss: 0.4386, Test Acc: 0.8024
Epoch [5/20] Train Loss: 0.2553, Train Acc: 0.9209 | Test Loss: 1.1194, Test Acc: 0.7082
Epoch [6/20] Train Loss: 0.2072, Train Acc: 0.9362 | Test Loss: 0.1749, Test Acc: 0.9473
Epoch [7/20] Train Loss: 0.1623, Train Acc: 0.9518 | Test Loss: 0.1187, Test Acc: 0.9671
Epoch [8/20] Train Loss: 0.0935, Train Acc: 0.9776 | Test Loss: 0.1145, Test Acc: 0.9710
Epoch [9/20] Train Loss: 0.0701, Train Acc: 0.9847 | Test Loss: 0.1092, Test Acc: 0.9696
Epoch [10/20] Train Loss: 0.0678, Train Acc: 0.9824 | Test Loss: 0.0530, Test Acc: 0.9870
Epoch [11/20] Train Loss: 0.0940, Train Acc: 0.9705 | Test Loss: 0.3007, Te

In [ ]:
# =========================
# Step 6: prediction example
# =========================

def predict(model, X_test_tensor, y_test, device):
    model.eval()

    with torch.no_grad():
        sample_x = X_test_tensor[:5].to(device)
        outputs = model(sample_x)
        preds = outputs.argmax(dim=1).cpu().numpy()

    print("Pred:", preds)
    print("True:", y_test[:5])

predict(model, X_test_r_tensor, y_test_r, device)

Pred: [1 2 1 0 2]
True: [1 2 1 0 2]


In [25]:
# =========================
# Step 7: confusion matrix
# =========================
from sklearn.metrics import confusion_matrix, classification_report

def compute_confusion_matrix(model, loader, device):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            outputs = model(xb)
            preds = outputs.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_true.extend(yb.numpy())

    cm = confusion_matrix(all_true, all_preds)
    print("Confusion Matrix:\n", cm)
    print("\nClassification Report:\n", classification_report(all_true, all_preds))

print("=== Version 1: random split ===\n")
compute_confusion_matrix(model, test_loader_r, device)

=== Version 1: random split ===

Confusion Matrix:
 [[597   5   0   0]
 [  0 499   0   0]
 [  0   0 471   0]
 [  0   0   0 498]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      1.00       602
           1       0.99      1.00      1.00       499
           2       1.00      1.00      1.00       471
           3       1.00      1.00      1.00       498

    accuracy                           1.00      2070
   macro avg       1.00      1.00      1.00      2070
weighted avg       1.00      1.00      1.00      2070



In [26]:
# ===========================
# version 2: env-based split
# ===========================

print("\n=== Version 2: env-based split ===\n")
train_and_evaluate(model, train_loader_e, test_loader_e, criterion, optimizer, device, num_epochs)


=== Version 2: env-based split ===

Epoch [1/20] Train Loss: 0.0115, Train Acc: 0.9985 | Test Loss: 0.0462, Test Acc: 0.9883
Epoch [2/20] Train Loss: 0.0030, Train Acc: 1.0000 | Test Loss: 0.0340, Test Acc: 0.9935
Epoch [3/20] Train Loss: 0.0031, Train Acc: 0.9998 | Test Loss: 0.0403, Test Acc: 0.9909
Epoch [4/20] Train Loss: 0.0028, Train Acc: 0.9998 | Test Loss: 0.0982, Test Acc: 0.9655
Epoch [5/20] Train Loss: 0.0017, Train Acc: 1.0000 | Test Loss: 0.1131, Test Acc: 0.9584
Epoch [6/20] Train Loss: 0.0017, Train Acc: 1.0000 | Test Loss: 0.0377, Test Acc: 0.9922
Epoch [7/20] Train Loss: 0.0013, Train Acc: 1.0000 | Test Loss: 0.0896, Test Acc: 0.9746
Epoch [8/20] Train Loss: 0.0014, Train Acc: 1.0000 | Test Loss: 0.0470, Test Acc: 0.9896
Epoch [9/20] Train Loss: 0.0011, Train Acc: 1.0000 | Test Loss: 0.0970, Test Acc: 0.9681
Epoch [10/20] Train Loss: 0.0014, Train Acc: 1.0000 | Test Loss: 0.0871, Test Acc: 0.9733
Epoch [11/20] Train Loss: 0.0009, Train Acc: 1.0000 | Test Loss: 0.1277,

In [ ]:
predict(model, X_test_e_tensor, y_test_e, device)

Pred: [3 3 3 3 3]
True: [0 0 0 0 0]


In [28]:
print("=== Version 2: env-based split ===\n")
compute_confusion_matrix(model, test_loader_e, device)

=== Version 2: env-based split ===

Confusion Matrix:
 [[379   0   0  24]
 [  0 454   0   0]
 [  0   0 358   0]
 [  4   0  29 290]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.94      0.96       403
           1       1.00      1.00      1.00       454
           2       0.93      1.00      0.96       358
           3       0.92      0.90      0.91       323

    accuracy                           0.96      1538
   macro avg       0.96      0.96      0.96      1538
weighted avg       0.96      0.96      0.96      1538

